# Lesson 2.1 — Selecting Claude Models by Trade-off

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Compare Claude model tiers on capability, latency, and cost
- Match a model tier to a task using an explicit scoring rubric
- Justify a model choice to a cost-conscious stakeholder

**You will produce:** A `model_selection.csv` scoring each task against a recommended Claude tier.

## Phase 2 Overview

Domain 2 (13%) covers model selection and the prompt/context engineering that steers Claude. This phase builds the decision tools architects use every day: a model-selection rubric, a reusable system-prompt template with guardrails, a prompt-technique comparison harness, and a token-budget/caching optimizer.

By the end of this phase, you will have built:

- A model-selection scoring sheet (CSV)
- A modular system-prompt template with guardrails
- A zero/few-shot/chain-of-thought comparison harness
- A token-budget model and a prompt-caching layout

### Prerequisites

- Completed Phase 1 (solution design and ADRs)
- Python 3 with pandas; `pip install anthropic pandas` (optional for live calls)
- Familiarity with tokens and context windows

## Concepts

Claude ships in tiers that trade **capability vs cost vs latency**:

- **Opus** — deepest reasoning; highest price and latency. Reserve for hard, high-value reasoning.
- **Sonnet** — balanced default; strong quality at moderate cost/latency. The usual workhorse.
- **Haiku** — fastest and cheapest; ideal for classification, routing, and high-volume simple tasks.

A **token** is the unit models read/write (~0.75 words). Pricing is per-token and differs by tier and by input vs output. The architect's move is to **right-size**: never pay Opus prices for a Haiku task, and never starve a complex reasoning task with an under-powered model ('model mismatch', a failure mode we diagnose in Phase 4). Always confirm current model IDs and prices against the official docs — they change.

> **Reference:** [Anthropic — Models overview](https://docs.anthropic.com/en/docs/about-claude/models)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

# Confirm exact model IDs at https://docs.anthropic.com/en/docs/about-claude/models
MODELS = {
    "opus": "claude-opus-4-20250514",
    "sonnet": "claude-sonnet-4-20250514",
    "haiku": "claude-3-5-haiku-20241022",
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Define a model-selection rubric

Score each task on reasoning depth, volume, and latency sensitivity, then recommend a tier. Keeping the rule explicit makes the choice defensible.

In [ ]:
def recommend_model(reasoning_depth, volume_per_day, latency_sensitive):
    """reasoning_depth: 1-5; volume_per_day: int; latency_sensitive: bool."""
    if reasoning_depth >= 4:
        return "opus", "Deep multi-step reasoning justifies the premium tier"
    if reasoning_depth <= 2 and (volume_per_day > 10000 or latency_sensitive):
        return "haiku", "Simple + high-volume/low-latency -> cheapest fast tier"
    return "sonnet", "Balanced default for moderate reasoning"

print(recommend_model(5, 100, False))
print(recommend_model(2, 50000, True))
print(recommend_model(3, 2000, False))

## Code-Along: Step 2 — Score a portfolio of tasks and export

Real systems run several tasks with different needs. We score a portfolio and export a CSV stakeholders can review — mixed-tier designs are normal and cost-smart.

In [ ]:
import csv
tasks = [
  ("intent routing", 1, 80000, True),
  ("policy reasoning", 5, 500, False),
  ("summarize ticket", 2, 20000, True),
  ("contract analysis", 4, 300, False),
]
rows = []
for name, depth, vol, lat in tasks:
    tier, why = recommend_model(depth, vol, lat)
    rows.append({"task": name, "tier": tier, "rationale": why})
with open("model_selection.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["task","tier","rationale"]); w.writeheader(); w.writerows(rows)
for r in rows: print(r)

## Exercise

Add a **budget guard**: given a monthly budget and rough per-task token estimates, flag any task whose recommended tier would blow the budget and suggest a downgrade. Extend `recommend_model` or wrap it. (Use placeholder per-1K-token prices and add a comment to verify current pricing at the Anthropic pricing docs.)

In [ ]:
# TODO: add placeholder prices per 1K tokens per tier (verify at https://docs.anthropic.com)
PRICE_PER_1K = {"haiku": 0.001, "sonnet": 0.003, "opus": 0.015}  # TODO verify
# TODO: estimate monthly cost per task and flag over-budget tasks

## Key Takeaways

- Right-size the model: match reasoning depth, volume, and latency to the tier.
- Mixed-tier architectures are normal — route cheap tasks to Haiku, hard ones to Opus.
- Model mismatch (too weak or too strong) is a real, diagnosable failure mode.
- Always re-verify model IDs and pricing against official docs before committing.

---

**Next:** [Lesson 2.2 — System Prompts, Templates, and Guardrails](./lesson-2.2-system-prompts-guardrails.ipynb)